# 04 · k vecinos más próximos (k-NN)

**Semana 3 de la planificación.** k-NN es el **aprendizaje basado en instancias** del tema 1: no aprende coeficientes, sino que guarda las canciones de entrenamiento y, para clasificar una canción nueva:

1. Calcula su distancia a todas las canciones de entrenamiento.
2. Se queda con las **k más cercanas** (las más parecidas).
3. Le asigna el género que más se repite entre esas k (votación).

Por eso es tan importante haber **estandarizado** los datos: la distancia suma diferencias de todas las características, y sin estandarizar las que tienen valores grandes dominarían el resultado.

El único hiperparámetro es **k**, y lo elegimos con el conjunto de validación. Usamos la implementación de scikit-learn porque el algoritmo es sencillo y lo que nos interesa es compararlo con la regresión logística.

In [ ]:
import sys, json
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from src.datos import cargar_procesados
from src.regresion_logistica import exactitud

CARPETA_FIGURAS = "../figuras"

datos = cargar_procesados("../datos/procesados.npz")
X_train, y_train = datos["X_train"], datos["y_train"]
X_val, y_val = datos["X_val"], datos["y_val"]
GENEROS = datos["generos"]

## 1. Curva de exactitud frente a k

In [ ]:
valores_k = [1, 3, 5, 7, 9, 15, 21, 31, 41, 51, 75, 101]
resultados = []

for k in valores_k:
    knn = KNeighborsClassifier(n_neighbors=k)   # distancia euclídea por defecto
    knn.fit(X_train, y_train)                   # "entrenar" = guardar las canciones
    resultados.append({
        "k": k,
        "exactitud train": exactitud(y_train, knn.predict(X_train)),
        "exactitud val": exactitud(y_val, knn.predict(X_val)),
    })

tabla_k = pd.DataFrame(resultados)
tabla_k.round(3)

In [ ]:
plt.figure(figsize=(8, 4.5))
plt.plot(tabla_k["k"], tabla_k["exactitud train"], "o-", label="entrenamiento")
plt.plot(tabla_k["k"], tabla_k["exactitud val"], "o-", label="validación")
plt.axhline(1 / 8, color="gray", linestyle="--", label="azar (12,5 %)")
plt.xlabel("k (número de vecinos)")
plt.ylabel("Exactitud")
plt.title("k-NN: efecto del número de vecinos")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CARPETA_FIGURAS}/04_curva_k.png", dpi=150)
plt.show()

**Cómo leer la curva:**

- **k = 1:** la exactitud en entrenamiento es del 100 %, porque el vecino más cercano de cada canción de entrenamiento es ella misma. No significa que el modelo sea bueno: es el caso extremo de **sobreajuste**.
- **k pequeño:** la frontera de decisión es muy irregular y sigue el ruido de los datos.
- **k muy grande:** se vota entre tantas canciones que el resultado se acerca al género más frecuente de la zona, y se pierde detalle (**subajuste**).
- El mejor k es el que maximiza la exactitud en validación.

## 2. Modelo final y matriz de confusión

In [ ]:
MEJOR_K = int(tabla_k.loc[tabla_k["exactitud val"].idxmax(), "k"])
knn = KNeighborsClassifier(n_neighbors=MEJOR_K).fit(X_train, y_train)
pred_val = knn.predict(X_val)
print(f"Mejor k: {MEJOR_K}  ->  exactitud en validación: {exactitud(y_val, pred_val):.3f}")

matriz = confusion_matrix(y_val, pred_val, labels=GENEROS)
fig, eje = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay(matriz, display_labels=GENEROS).plot(ax=eje, cmap="Greens", colorbar=False)
plt.xticks(rotation=40, ha="right")
eje.set_xlabel("Género predicho")
eje.set_ylabel("Género real")
eje.set_title(f"k-NN (k = {MEJOR_K}) · validación")
plt.tight_layout()
plt.savefig(f"{CARPETA_FIGURAS}/04_matriz_confusion_knn.png", dpi=150)
plt.show()

## 3. Comparación preliminar con la regresión logística

Comparamos los dos modelos **en validación**. Es una comparación preliminar: la definitiva se hará en test al final del proyecto, después de la selección de características con el algoritmo genético.

In [ ]:
with open("../resultados/regresion_logistica.json") as f:
    logistica = json.load(f)

aciertos_knn = pd.Series(np.diag(matriz) / matriz.sum(axis=1), index=GENEROS)
comparacion = pd.DataFrame({
    "Regresión logística": pd.Series(logistica["aciertos_por_genero"]),
    "k-NN": aciertos_knn,
})
comparacion.loc["TOTAL (exactitud)"] = [logistica["exactitud_val"], exactitud(y_val, pred_val)]
comparacion.round(3)

In [ ]:
por_genero = comparacion.drop("TOTAL (exactitud)")
posiciones = np.arange(len(por_genero))

plt.figure(figsize=(10, 4.5))
plt.bar(posiciones - 0.2, por_genero["Regresión logística"], width=0.4, label=f"Regresión logística (λ = {logistica['mejor_lambda']})")
plt.bar(posiciones + 0.2, por_genero["k-NN"], width=0.4, label=f"k-NN (k = {MEJOR_K})")
plt.xticks(posiciones, por_genero.index, rotation=30, ha="right")
plt.ylabel("Proporción de aciertos")
plt.title("Aciertos por género en validación")
plt.legend()
plt.tight_layout()
plt.savefig(f"{CARPETA_FIGURAS}/04_comparacion_modelos.png", dpi=150)
plt.show()

In [ ]:
with open("../resultados/knn.json", "w") as f:
    json.dump({"mejor_k": MEJOR_K, "exactitud_val": float(exactitud(y_val, pred_val)),
               "aciertos_por_genero": {g: float(v) for g, v in aciertos_knn.items()}}, f, indent=2, ensure_ascii=False)
tabla_k.to_csv("../resultados/curva_k.csv", index=False)
print("Resultados guardados en la carpeta resultados/")

**Relación con la hipótesis H2:** esperábamos que, con las 518 características, la regresión logística superase a k-NN. El motivo es que k-NN da el mismo peso a todas las características en la distancia, incluidas las que no aportan nada, mientras que la regresión logística puede asignar coeficientes pequeños a las inútiles. Si se cumple, es una pista de que la selección de características (algoritmo genético, semana 4) debería ayudar sobre todo a k-NN (**hipótesis H3**).

## Resumen

- k-NN clasifica por parecido con las canciones de entrenamiento, sin construir un modelo explícito.
- La curva frente a k muestra de nuevo el equilibrio entre sobreajuste y subajuste, y elegimos k con validación.
- Tenemos una primera comparación entre un método basado en modelos y otro basado en instancias, que es el punto de partida para el algoritmo genético.